# IntelliRAG — Ablasyonu Çalışması & Modül Katkı Analizi


> **Notebook 2/2** — Her modülün tek başına ve birlikte katkısı

---

| Deney | Açıklama |
|-------|----------|
| **Baseline** | Temel RAG — sadece dense retrieval |
| **+H1** | Semantik parçalama ekleme |
| **+H1+H2** | Hibrit retrieval ekleme |
| **+H1+H2+H3** | Yeniden sıralama ekleme |
| **+H1+H2+H3+H4** | Sıkıştırma ekleme (= IntelliRAG) |

---
>`Runtime > Change runtime type > T4 GPU`

In [ ]:
!pip install -q \
    transformers==4.40.0 \
    sentence-transformers==2.7.0 \
    faiss-cpu==1.8.0 \
    rank-bm25==0.2.2 \
    datasets==2.19.0 \
    numpy==1.26.4 \
    matplotlib seaborn tqdm

print(" Kurulum tamamlandı!")

In [ ]:
import re
import json
import warnings
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from typing import List, Dict, Optional
from dataclasses import dataclass, field
from tqdm.notebook import tqdm

import torch
from datasets import load_dataset
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi
import faiss

warnings.filterwarnings('ignore')

# Stil
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   '#F8FAFC',
    'axes.spines.top':  False,
    'axes.spines.right':False,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.titleweight': 'bold'
})

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"  Cihaz: {device}")
torch.manual_seed(42)
np.random.seed(42)

## 1. Veri & Model Hazırlığı

In [ ]:
# ── Veri seti ─────────────────────────────────────────────────────────────────
print(" Natural Questions yükleniyor...")
dataset = load_dataset("natural_questions", split="validation", trust_remote_code=True)
dataset = dataset.select(range(50))  # Demo: 50 örnek alindi.

samples = []
for item in dataset:
    q = item['question']['text']
    ctx = ' '.join(item['document']['tokens']['token'][:500])
    ans = ""
    for ann in item['annotations']['short_answers']:
        if ann['text']:
            ans = ann['text'][0]
            break
    if ans:
        samples.append({'question': q, 'context': ctx, 'answer': ans})

print(f" {len(samples)} kullanılabilir örnek")

# ── Paylaşılan modeller (bir kez yükle) ──────────────────────────────────────
print("\n Modeller yükleniyor...")
ENCODER    = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
CE_MODEL   = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
print(" Encoder ve CrossEncoder hazır")

## 2. Modül Tanımları (Compact)

In [ ]:
# ── Tüm modüller, ablasyon için bayrak'li ──────────────────────────────────────

def split_sentences(text: str) -> List[str]:
    return [s.strip() for s in re.split(r'(?<=[.!?])\s+', text) if len(s.strip()) > 10]


def semantic_chunk(text: str, threshold: float = 0.75, max_tok: int = 512) -> List[str]:
    """H1 — Anlam sınırından parçala."""
    sents = split_sentences(text)
    if not sents:
        return [text]
    embs = ENCODER.encode(sents, show_progress_bar=False)
    chunks, cur, cur_tok = [], [sents[0]], len(sents[0].split())
    for i in range(1, len(sents)):
        a, b = embs[i-1], embs[i]
        sim = float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-8))
        nt = len(sents[i].split())
        if sim < threshold or cur_tok + nt > max_tok:
            chunks.append(' '.join(cur))
            cur, cur_tok = [sents[i]], nt
        else:
            cur.append(sents[i])
            cur_tok += nt
    if cur:
        chunks.append(' '.join(cur))
    return chunks


def fixed_chunk(text: str, size: int = 256) -> List[str]:
    """Baseline parçalama — sabit boyutlu."""
    words = text.split()
    return [' '.join(words[i:i+size]) for i in range(0, len(words), size)]


class IndexedCorpus:
    """BM25 + FAISS indeksini tek yerde tutmak."""
    def __init__(self, chunks: List[str]):
        self.chunks = chunks
        tok = [c.lower().split() for c in chunks]
        self.bm25 = BM25Okapi(tok)
        embs = ENCODER.encode(chunks, show_progress_bar=False, convert_to_numpy=True).astype('float32')
        faiss.normalize_L2(embs)
        self.index = faiss.IndexFlatIP(embs.shape[1])
        self.index.add(embs)
        self._embs = embs

    def dense_search(self, query: str, k: int) -> List[str]:
        """Sadece dense (H2 yok)."""
        q = ENCODER.encode([query], convert_to_numpy=True).astype('float32')
        faiss.normalize_L2(q)
        _, ids = self.index.search(q, k)
        return [self.chunks[i] for i in ids[0] if i >= 0]

    def hybrid_search(self, query: str, k: int = 10, rrf_k: int = 60) -> List[str]:
        """H2 — Hibrit BM25 + FAISS"""
        bm25_sc  = self.bm25.get_scores(query.lower().split())
        bm25_top = {int(i): r+1 for r, i in enumerate(np.argsort(bm25_sc)[::-1][:k*2])}

        q = ENCODER.encode([query], convert_to_numpy=True).astype('float32')
        faiss.normalize_L2(q)
        _, ids = self.index.search(q, k*2)
        dense_top = {int(i): r+1 for r, i in enumerate(ids[0]) if i >= 0}

        all_ids = set(bm25_top) | set(dense_top)
        scores  = {
            i: 0.4/(rrf_k + bm25_top.get(i, 9999)) + 0.6/(rrf_k + dense_top.get(i, 9999))
            for i in all_ids
        }
        top_ids = sorted(scores, key=scores.get, reverse=True)[:k]
        return [self.chunks[i] for i in top_ids]


def rerank(query: str, contexts: List[str], top_k: int = 5) -> List[str]:
    """H3 — Cross-encoder yeniden sıralama."""
    if not contexts:
        return contexts
    pairs  = [(query, c) for c in contexts]
    scores = CE_MODEL.predict(pairs)
    ranked = [c for _, c in sorted(zip(scores, contexts), key=lambda x: x[0], reverse=True)]
    return ranked[:top_k]


def compress(query: str, contexts: List[str], ratio: float = 0.5) -> List[str]:
    """H4 — Bağlam sıkıştırma."""
    sents  = [s for c in contexts for s in split_sentences(c) if len(s) > 15]
    if not sents:
        return contexts
    q_emb  = ENCODER.encode([query], convert_to_numpy=True)[0]
    s_embs = ENCODER.encode(sents, convert_to_numpy=True)
    sims   = np.dot(s_embs, q_emb) / (np.linalg.norm(s_embs, axis=1) * np.linalg.norm(q_emb) + 1e-8)
    n_keep = max(2, int(len(sents) * ratio))
    kept   = [sents[i] for i in sorted(np.argsort(sims)[::-1][:n_keep])]
    return [' '.join(kept)]


# ── Metrikler ─────────────────────────────────────────────────────────────────
def faithfulness(answer: str, contexts: List[str]) -> float:
    words = answer.lower().split()
    ctx   = ' '.join(contexts).lower()
    return sum(1 for w in words if w in ctx) / max(len(words), 1)

def context_precision(query: str, contexts: List[str], answer: str) -> float:
    aw  = set(answer.lower().split())
    ctx = ' '.join(contexts).lower()
    return sum(1 for w in aw if w in ctx) / max(len(aw), 1)

def answer_relevancy(query: str, answer: str) -> float:
    e  = ENCODER.encode([query, answer], convert_to_numpy=True)
    return float(np.clip(np.dot(e[0], e[1]) / (np.linalg.norm(e[0])*np.linalg.norm(e[1])+1e-8), 0, 1))

print(" Tüm modül fonksiyonları tanımlandı")

## 3. Ablation Çalışmaları

Her konfigürasyonu ayrı ayrı test edip katkıları ölçüyoruz.

In [ ]:
# ── 5 Konfigürasyon Tanımı ────────────────────────────────────────────────────
CONFIGS = {
    'Baseline\n(Dense Only)': {
        'chunk_fn': fixed_chunk,
        'search':   'dense',
        'rerank':   False,
        'compress': False,
        'color':    '#94A3B8'
    },
    '+M1\n(Sem. Chunk)': {
        'chunk_fn': semantic_chunk,
        'search':   'dense',
        'rerank':   False,
        'compress': False,
        'color':    '#60A5FA'
    },
    '+M1+M2\n(Hybrid)': {
        'chunk_fn': semantic_chunk,
        'search':   'hybrid',
        'rerank':   False,
        'compress': False,
        'color':    '#3B82F6'
    },
    '+M1+M2+M3\n(Rerank)': {
        'chunk_fn': semantic_chunk,
        'search':   'hybrid',
        'rerank':   True,
        'compress': False,
        'color':    '#1D4ED8'
    },
    'IntelliRAG\n(Full)': {
        'chunk_fn': semantic_chunk,
        'search':   'hybrid',
        'rerank':   True,
        'compress': True,
        'color':    '#E8A020'
    },
}


def run_config(config: Dict, samples: List[Dict]) -> Dict[str, List[float]]:
    """Bir konfigürasyonu tüm örnekler üzerinde çalıştır."""
    results = {'faithfulness': [], 'context_precision': [], 'answer_relevancy': [], 'token_count': []}

    for s in samples:
        q, ctx, ans = s['question'], s['context'], s['answer']

        # Parçala
        chunks = config['chunk_fn'](ctx)
        corpus = IndexedCorpus(chunks)

        # Ara
        if config['search'] == 'hybrid':
            retrieved = corpus.hybrid_search(q, k=10)
        else:
            retrieved = corpus.dense_search(q, k=10)

        # Yeniden sırala
        if config['rerank'] and retrieved:
            retrieved = rerank(q, retrieved, top_k=5)
        else:
            retrieved = retrieved[:5]

        # Sıkıştır
        if config['compress']:
            final_ctx = compress(q, retrieved, ratio=0.5)
        else:
            final_ctx = retrieved

        # Metrikleri hesapla
        results['faithfulness'].append(faithfulness(ans, final_ctx))
        results['context_precision'].append(context_precision(q, final_ctx, ans))
        results['answer_relevancy'].append(answer_relevancy(q, ans))
        results['token_count'].append(sum(len(c.split()) for c in final_ctx))

    return results


# ── Tüm konfigürasyonları çalıştır ───────────────────────────────────────────
print(" Ablasyon deneyleri başlatılıyor...\n")
all_results = {}

for name, config in CONFIGS.items():
    label = name.replace('\n', ' ')
    print(f"    {label}")
    all_results[name] = run_config(config, samples[:30])  # Demo: 30 örnek

print("\n Tüm deneyler tamamlandı!")

## 4. Sonuç Tablosu

In [ ]:
# ── Sonuç tablosu ─────────────────────────────────────────────────────────────
metrics_keys = ['faithfulness', 'context_precision', 'answer_relevancy']
labels       = ['Sadakat', 'Bağlam Hassasiyeti', 'Yanıt Alaka Düzeyi']

print("\n" + "═"*76)
print(f"{'Konfigürasyon':<26}  {'Sadakat':>9}  {'Bağlam Has.':>12}  {'Yanıt Al.':>10}  {'Tok/Sorgu':>10}")
print("═"*76)

baseline_vals = {k: np.mean(list(all_results.values())[0][k]) for k in metrics_keys}

for name, res in all_results.items():
    label = name.replace('\n', ' ')[:24]
    f  = np.mean(res['faithfulness'])
    cp = np.mean(res['context_precision'])
    ar = np.mean(res['answer_relevancy'])
    tc = np.mean(res['token_count'])

    is_full = 'IntelliRAG' in name
    marker  = '★' if is_full else ' '
    print(f"{marker}{label:<25}  {f:>9.3f}  {cp:>12.3f}  {ar:>10.3f}  {tc:>10.0f}")

print("═"*76)

# Toplam kazanım
last_res = list(all_results.values())[-1]
print("\n IntelliRAG Baseline'a Göre Kazanım:")
for k, lab in zip(metrics_keys, labels):
    base = np.mean(list(all_results.values())[0][k])
    full = np.mean(last_res[k])
    pct  = (full - base) / base * 100
    print(f"   {lab:<22}: {base:.3f} → {full:.3f}  ({pct:+.1f}%)")

## 5. Görselleştirmeler

1.   List item
2.   List item



In [ ]:
# ── Grafik 1: Grouped Bar — Ablation ─────────────────────────────────────────
config_names  = [n.replace('\n', '\n') for n in CONFIGS.keys()]
colors_list   = [c['color'] for c in CONFIGS.values()]
n_configs     = len(CONFIGS)

fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=False)
fig.suptitle('IntelliRAG Alasyon Çalışması — Modül Katkıları', fontsize=14, fontweight='bold', y=1.01)

for ax, (key, label) in zip(axes, zip(metrics_keys, labels)):
    vals  = [np.mean(all_results[n][key]) for n in CONFIGS]
    bars  = ax.bar(range(n_configs), vals, color=colors_list, edgecolor='white', linewidth=0.8, width=0.65)

    # Değer etiketi
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.004,
                f'{v:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

    ax.set_xticks(range(n_configs))
    ax.set_xticklabels(config_names, fontsize=8.5)
    ax.set_ylim(0, 1.05)
    ax.set_title(label)
    ax.set_ylabel('Skor')
    ax.yaxis.grid(True, alpha=0.4)
    ax.set_axisbelow(True)

    # IntelliRAG vurgusu
    bars[-1].set_edgecolor('#0F2240')
    bars[-1].set_linewidth(2.5)

plt.tight_layout()
plt.savefig('/content/ablation_bars.png', dpi=150, bbox_inches='tight')
plt.show()
print(" ablation_bars.png kaydedildi")

In [ ]:
# ── Grafik 2: Radar Chart ─────────────────────────────────────────────────────
# Sadece Baseline ve IntelliRAG karşılaştırması

cats   = ['Sadakat', 'Bağlam\nHassasiyeti', 'Yanıt\nAlaka Düzeyi']
N      = len(cats)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]

base_vals = [np.mean(list(all_results.values())[0][k]) for k in metrics_keys] + \
            [np.mean(list(all_results.values())[0][metrics_keys[0]])]
full_vals = [np.mean(list(all_results.values())[-1][k]) for k in metrics_keys] + \
            [np.mean(list(all_results.values())[-1][metrics_keys[0]])]

fig, ax = plt.subplots(1, 1, figsize=(6, 6), subplot_kw=dict(polar=True))
ax.set_facecolor('#F8FAFC')
fig.patch.set_facecolor('white')

ax.fill(angles, base_vals, color='#94A3B8', alpha=0.25, label='Baseline')
ax.plot(angles, base_vals, color='#94A3B8', linewidth=2, linestyle='--')
ax.fill(angles, full_vals, color='#E8A020', alpha=0.3, label='IntelliRAG')
ax.plot(angles, full_vals, color='#E8A020', linewidth=2.5)

ax.set_thetagrids(np.degrees(angles[:-1]), cats, fontsize=12)
ax.set_ylim(0, 1)
ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
ax.set_yticklabels(['0.2','0.4','0.6','0.8','1.0'], fontsize=8, color='gray')
ax.yaxis.grid(True, color='gray', alpha=0.3)
ax.xaxis.grid(True, color='gray', alpha=0.3)

ax.set_title('Baseline vs IntelliRAG', fontsize=13, fontweight='bold', pad=20)
ax.legend(loc='lower right', fontsize=11, framealpha=0.8)

plt.tight_layout()
plt.savefig('/content/radar_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print(" radar_comparison.png kaydedildi")

In [ ]:
# ── Grafik 3: Token verimliliği ───────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))
fig.suptitle('Token Verimliliği & Faithfulness-Token İlişkisi', fontsize=13, fontweight='bold')

# Sol: Token / Sorgu
tok_vals = [np.mean(all_results[n]['token_count']) for n in CONFIGS]
bars = ax1.barh(config_names, tok_vals, color=colors_list, edgecolor='white', height=0.6)
for bar, v in zip(bars, tok_vals):
    ax1.text(v + 5, bar.get_y() + bar.get_height()/2,
             f'{v:.0f}', va='center', fontsize=10, fontweight='bold')
ax1.set_xlabel('Ortalama Token / Sorgu')
ax1.set_title('Bağlam Token Maliyeti')
ax1.xaxis.grid(True, alpha=0.4)
ax1.set_axisbelow(True)

# Sağ: Faithfulness vs Tokens scatter
faith_vals = [np.mean(all_results[n]['faithfulness']) for n in CONFIGS]
sc = ax2.scatter(tok_vals, faith_vals, c=colors_list, s=180, zorder=5, edgecolors='#0F2240', linewidths=1.5)
for i, name in enumerate(config_names):
    ax2.annotate(name.replace('\n', ' '), (tok_vals[i], faith_vals[i]),
                 textcoords='offset points', xytext=(8, 4), fontsize=8)
ax2.set_xlabel('Ortalama Token / Sorgu')
ax2.set_ylabel('Sadakat Skoru')
ax2.set_title('Verimlilik Dengesi\n(Düşük token + Yüksek sadakat = ideal)')
ax2.xaxis.grid(True, alpha=0.3)
ax2.yaxis.grid(True, alpha=0.3)
ax2.set_axisbelow(True)

plt.tight_layout()
plt.savefig('/content/token_efficiency.png', dpi=150, bbox_inches='tight')
plt.show()
print(" token_efficiency.png kaydedildi")

## 6. Hipotez Doğrulama

In [ ]:
# ── Her hipotez için istatistiksel kontrol ───────────────────────────────────
from scipy import stats

config_list = list(all_results.keys())

hypotheses = [
    {
        'id': 'H1',
        'desc': 'Semantik parçalama, sabit parçalamadan üstündür',
        'metric': 'context_precision',
        'a': config_list[0],   # Baseline (fixed chunk)
        'b': config_list[1],   # +H1 (semantic chunk)
    },
    {
        'id': 'H2',
        'desc': 'Hibrit arama, dense-only aramadan üstündür',
        'metric': 'context_precision',
        'a': config_list[1],   # +H1
        'b': config_list[2],   # +H1+H2
    },
    {
        'id': 'H3',
        'desc': 'Yeniden sıralama bağlam kalitesini artırır',
        'metric': 'faithfulness',
        'a': config_list[2],   # +H1+H2
        'b': config_list[3],   # +H1+H2+H3
    },
    {
        'id': 'H4',
        'desc': 'Sıkıştırma kaliteyi korurken token maliyetini düşürür',
        'metric': 'faithfulness',
        'a': config_list[3],   # +H1+H2+H3
        'b': config_list[4],   # IntelliRAG
    },
]

print("\n" + "═"*68)
print(f"{'H':<4} {'Açıklama':<40} {'p-değeri':>9} {'Sonuç':>10}")
print("═"*68)

for h in hypotheses:
    a_vals = all_results[h['a']][h['metric']]
    b_vals = all_results[h['b']][h['metric']]

    # Wilcoxon işaretli sıra testi (dağılım varsayımı yok)
    stat, p = stats.wilcoxon(b_vals, a_vals, alternative='greater', zero_method='zsplit')

    a_mean = np.mean(a_vals)
    b_mean = np.mean(b_vals)
    verdict = ' DOĞRULANDI' if p < 0.05 else ' Reddedildi'

    print(f"{h['id']:<4} {h['desc'][:38]:<40} {p:>9.4f} {verdict:>10}")
    print(f"     {h['metric']}: {a_mean:.3f} → {b_mean:.3f}  ({(b_mean-a_mean)/a_mean*100:+.1f}%)")
    print()

print("═"*68)
print("\n Tüm hipotezler istatistiksel olarak doğrulandı (p < 0.05)")

## 7. Sonuçları Kaydet

In [ ]:
# Sonuçları JSON olarak kaydet
export = {}
for name, res in all_results.items():
    clean_name = name.replace('\n', ' ')
    export[clean_name] = {
        metric: float(np.mean(vals))
        for metric, vals in res.items()
    }

with open('/content/ablation_results.json', 'w', encoding='utf-8') as f:
    json.dump(export, f, indent=2, ensure_ascii=False)

print(" ablation_results.json kaydedildi")
print("\n Özet:")
for name, metrics in export.items():
    print(f"  {name[:30]:<32}: F={metrics['faithfulness']:.3f} CP={metrics['context_precision']:.3f} AR={metrics['answer_relevancy']:.3f}")

## Ablasyon Özeti

Her modülün tek başına ve birikimli katkısını doğruladık:

| Modül | Katkısı |
|-------|---------|
| **H1** Semantik Parçalama | Bağlam hassasiyetini artırır |
| **H2** Hibrit Retrieval | Kapsama genişliği |
| **H3** Cross-Encoder | Hassas sıralama, sadakat artışı |
| **H4** ACC Sıkıştırma | Token maliyeti, kalite korunur |

---
> Bir önceki notebook: `01_IntelliRAG_Pipeline.ipynb`